In [1]:
# CELL 1 — IMPORT LIBRARIES AND SET PROJECT FOLDERS

from pathlib import Path
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data folder is created beside this notebook.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

REGISTRY_DIR = DATA_DIR / "geographic_registry"
INPUT_DIR = DATA_DIR / "input"
TIER1_DIR = INPUT_DIR / "tier1_country"
TIER2_DIR = INPUT_DIR / "tier2_region"
TIER3_DIR = INPUT_DIR / "tier3_city"
OUTPUT_DIR = DATA_DIR / "output"

for folder in [REGISTRY_DIR, TIER1_DIR, TIER2_DIR, TIER3_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Place your indicator CSV files in:")
print("Tier 1:", TIER1_DIR)
print("Tier 2:", TIER2_DIR)
print("Tier 3:", TIER3_DIR)

Project folder: /content
Place your indicator CSV files in:
Tier 1: /content/data/input/tier1_country
Tier 2: /content/data/input/tier2_region
Tier 3: /content/data/input/tier3_city


In [2]:
# CELL 2 — NESTED CRITERIA DICTIONARY

CRITERIA = {
    "tier1_country": {
        "healthcare": {
            "physicians_per_1000": {
                "direction": "benefit",
                "unit": "physicians per 1,000 people",
                "description": "Availability of physicians at the country level"
            },
            "health_expenditure_per_capita_ppp": {
                "direction": "benefit",
                "unit": "international dollars per person",
                "description": "Health expenditure per person, PPP-adjusted"
            }
        },
        "education": {
            "secondary_completion_rate": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Share of the relevant population completing secondary education"
            },
            "tertiary_education_rate": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Share of adults with tertiary education"
            }
        },
        "economics_and_opportunity": {
            "gdp_per_capita_ppp": {
                "direction": "benefit",
                "unit": "international dollars per person",
                "description": "GDP per capita adjusted for purchasing power"
            },
            "unemployment_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "National unemployment rate"
            },
            "poverty_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "Share of population below a consistently defined poverty threshold"
            }
        },
        "family_and_social_conditions": {
            "median_household_income_ppp": {
                "direction": "benefit",
                "unit": "international dollars",
                "description": "Median household disposable income, PPP-adjusted"
            },
            "housing_cost_burden": {
                "direction": "cost",
                "unit": "percent",
                "description": "Share of household income spent on housing"
            }
        }
    },

    "tier2_region": {
        "regional_healthcare": {
            "regional_hospital_beds_per_1000": {
                "direction": "benefit",
                "unit": "beds per 1,000 people",
                "description": "Hospital bed availability in the region"
            },
            "regional_physicians_per_1000": {
                "direction": "benefit",
                "unit": "physicians per 1,000 people",
                "description": "Physician availability in the region"
            }
        },
        "regional_economics": {
            "regional_unemployment_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "Unemployment rate for the administrative region"
            },
            "regional_median_income_ppp": {
                "direction": "benefit",
                "unit": "international dollars",
                "description": "Regional median disposable income, adjusted for purchasing power"
            },
            "regional_housing_cost_burden": {
                "direction": "cost",
                "unit": "percent",
                "description": "Regional housing costs relative to household income"
            }
        },
        "safety_and_environment": {
            "regional_homicide_rate": {
                "direction": "cost",
                "unit": "per 100,000 people",
                "description": "Homicides per 100,000 residents; use a consistent definition"
            },
            "regional_pm25": {
                "direction": "cost",
                "unit": "micrograms per cubic meter",
                "description": "Annual mean fine particulate matter concentration"
            },
            "regional_green_space_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Population with access to green space under one consistent definition"
            }
        }
    },

    "tier3_city": {
        "local_access": {
            "city_healthcare_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Residents within a defined travel time of primary healthcare"
            },
            "public_transport_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Residents within a defined walk distance of public transport"
            }
        },
        "local_safety_and_environment": {
            "city_homicide_rate": {
                "direction": "cost",
                "unit": "per 100,000 people",
                "description": "City homicide rate using a consistent reporting definition"
            },
            "city_pm25": {
                "direction": "cost",
                "unit": "micrograms per cubic meter",
                "description": "Annual mean city-level PM2.5 concentration"
            },
            "city_green_space_per_capita": {
                "direction": "benefit",
                "unit": "square meters per person",
                "description": "Public green space per resident"
            }
        },
        "amenities_and_diversity": {
            "restaurants_per_10000": {
                "direction": "benefit",
                "unit": "restaurants per 10,000 people",
                "description": "Restaurant availability, ideally from a consistent POI source"
            },
            "cultural_venues_per_10000": {
                "direction": "benefit",
                "unit": "venues per 10,000 people",
                "description": "Museums, theatres, libraries, and cultural venues per capita"
            },
            "diversity_index": {
                "direction": "benefit",
                "unit": "0 to 1",
                "description": "A documented demographic diversity index"
            }
        }
    }
}

# Validate that all directions are entered correctly.
for tier_name, categories in CRITERIA.items():
    for category_name, variables in categories.items():
        for criterion, details in variables.items():
            assert details["direction"] in ["benefit", "cost"], (
                f"Invalid direction for {criterion}"
            )

print("Criteria successfully loaded.")
for tier_name, categories in CRITERIA.items():
    all_keys = [key for group in categories.values() for key in group]
    print(f"{tier_name}: {len(all_keys)} criteria -> {all_keys}")

Criteria successfully loaded.
tier1_country: 9 criteria -> ['physicians_per_1000', 'health_expenditure_per_capita_ppp', 'secondary_completion_rate', 'tertiary_education_rate', 'gdp_per_capita_ppp', 'unemployment_rate', 'poverty_rate', 'median_household_income_ppp', 'housing_cost_burden']
tier2_region: 8 criteria -> ['regional_hospital_beds_per_1000', 'regional_physicians_per_1000', 'regional_unemployment_rate', 'regional_median_income_ppp', 'regional_housing_cost_burden', 'regional_homicide_rate', 'regional_pm25', 'regional_green_space_access']
tier3_city: 8 criteria -> ['city_healthcare_access', 'public_transport_access', 'city_homicide_rate', 'city_pm25', 'city_green_space_per_capita', 'restaurants_per_10000', 'cultural_venues_per_10000', 'diversity_index']


In [3]:
# CELL 3 — CREATE BLANK CSV TEMPLATES FOR EVERY CRITERION

REQUIRED_COLUMNS = ["geo_id", "value", "year", "source", "quality"]

def create_csv_templates():
    for tier_name, categories in CRITERIA.items():
        folder = {
            "tier1_country": TIER1_DIR,
            "tier2_region": TIER2_DIR,
            "tier3_city": TIER3_DIR
        }[tier_name]

        for category_name, variables in categories.items():
            for criterion_key in variables:
                file_path = folder / f"{criterion_key}.csv"
                if not file_path.exists():
                    template = pd.DataFrame(columns=REQUIRED_COLUMNS)
                    if tier_name != "tier1_country":
                        template["country_id"] = pd.Series(dtype="str")
                    if tier_name == "tier3_city":
                        template["region_id"] = pd.Series(dtype="str")
                    template.to_csv(file_path, index=False)

create_csv_templates()
print("Blank criterion CSV templates created (existing files were not overwritten).")

Blank criterion CSV templates created (existing files were not overwritten).


In [4]:
# CELL 3 — CREATE BLANK CSV TEMPLATES FOR EVERY CRITERION

REQUIRED_COLUMNS = ["geo_id", "value", "year", "source", "quality"]

def create_csv_templates():
    for tier_name, categories in CRITERIA.items():
        folder = {
            "tier1_country": TIER1_DIR,
            "tier2_region": TIER2_DIR,
            "tier3_city": TIER3_DIR
        }[tier_name]

        for category_name, variables in categories.items():
            for criterion_key in variables:
                file_path = folder / f"{criterion_key}.csv"
                if not file_path.exists():
                    template = pd.DataFrame(columns=REQUIRED_COLUMNS)
                    if tier_name != "tier1_country":
                        template["country_id"] = pd.Series(dtype="str")
                    if tier_name == "tier3_city":
                        template["region_id"] = pd.Series(dtype="str")
                    template.to_csv(file_path, index=False)

create_csv_templates()
print("Blank criterion CSV templates created (existing files were not overwritten).")

Blank criterion CSV templates created (existing files were not overwritten).


In [5]:
# CELL 4 — DOWNLOAD COUNTRY, REGION, AND CITY REGISTRIES

GEONAMES_BASE = "https://download.geonames.org/export/dump"

def download_if_missing(url, destination):
    destination = Path(destination)
    if not destination.exists():
        print("Downloading:", url)
        response = requests.get(url, timeout=180)
        response.raise_for_status()
        destination.write_bytes(response.content)
    return destination

country_file = download_if_missing(
    f"{GEONAMES_BASE}/countryInfo.txt",
    REGISTRY_DIR / "countryInfo.txt"
)
region_file = download_if_missing(
    f"{GEONAMES_BASE}/admin1CodesASCII.txt",
    REGISTRY_DIR / "admin1CodesASCII.txt"
)
city_zip_file = download_if_missing(
    f"{GEONAMES_BASE}/cities1000.zip",
    REGISTRY_DIR / "cities1000.zip"
)

print("Registry source files are ready.")

Downloading: https://download.geonames.org/export/dump/countryInfo.txt
Downloading: https://download.geonames.org/export/dump/admin1CodesASCII.txt
Downloading: https://download.geonames.org/export/dump/cities1000.zip
Registry source files are ready.


In [6]:
# CELL 5 — READ THE GEOGRAPHIC REGISTRIES

# GeoNames countryInfo fields (comment lines are skipped).
country_columns = [
    "ISO2", "ISO3", "ISO_numeric", "fips", "country_name",
    "capital", "area_sq_km", "population", "continent",
    "tld", "currency_code", "currency_name", "phone",
    "postal_code_format", "postal_code_regex", "languages",
    "geonameid", "neighbours", "equivalent_fips"
]

countries = pd.read_csv(
    country_file, sep="\t", comment="#", header=None,
    names=country_columns, dtype={"ISO2": str, "ISO3": str}
)

# First-level administrative regions: code, name, ASCII name, GeoNames ID.
regions = pd.read_csv(
    region_file, sep="\t", header=None,
    names=["region_id", "region_name", "region_ascii", "region_geoname_id"],
    dtype={"region_id": str}
)
regions["country_id"] = regions["region_id"].str.split(".").str[0]

# GeoNames cities1000 file has 19 tab-separated columns.
city_columns = [
    "geoname_id", "city_name", "ascii_name", "alternate_names",
    "latitude", "longitude", "feature_class", "feature_code",
    "country_id", "alternate_country_codes", "admin1_code",
    "admin2_code", "admin3_code", "admin4_code", "population",
    "elevation", "dem", "timezone", "modification_date"
]

with zipfile.ZipFile(city_zip_file) as zf:
    city_member = next(name for name in zf.namelist() if name.endswith(".txt"))
    cities = pd.read_csv(
        zf.open(city_member), sep="\t", header=None,
        names=city_columns, dtype={
            "geoname_id": str, "country_id": str, "admin1_code": str
        },
        low_memory=False
    )

# Build a consistent first-level region ID, such as US.MI.
cities["region_id"] = np.where(
    cities["admin1_code"].notna() & (cities["admin1_code"] != ""),
    cities["country_id"] + "." + cities["admin1_code"],
    np.nan
)
cities["geo_id"] = cities["geoname_id"].astype(str)

# Keep populated-place features and useful identifying fields.
cities = cities[cities["feature_class"] == "P"].copy()
cities["population"] = pd.to_numeric(cities["population"], errors="coerce")
cities = cities.dropna(subset=["country_id", "latitude", "longitude"])

country_registry = countries[[
    "ISO2", "ISO3", "country_name", "continent", "population"
]].rename(columns={
    "ISO2": "geo_id",
    "country_name": "place_name"
}).dropna(subset=["geo_id"]).drop_duplicates("geo_id")

region_registry = regions[[
    "region_id", "region_name", "country_id", "region_geoname_id"
]].rename(columns={
    "region_id": "geo_id",
    "region_name": "place_name"
}).drop_duplicates("geo_id")

city_registry = cities[[
    "geo_id", "city_name", "country_id", "region_id",
    "latitude", "longitude", "population", "timezone"
]].rename(columns={"city_name": "place_name"}).drop_duplicates("geo_id")

country_registry.to_csv(REGISTRY_DIR / "countries.csv", index=False)
region_registry.to_csv(REGISTRY_DIR / "regions.csv", index=False)
city_registry.to_csv(REGISTRY_DIR / "cities_geonames1000.csv", index=False)

print("Countries/areas:", len(country_registry))
print("Administrative regions:", len(region_registry))
print("Populated-place records:", len(city_registry))

Countries/areas: 251
Administrative regions: 3865
Populated-place records: 171726


In [7]:
# CELL 6 — SCREENING SETTINGS

TOP_COUNTRIES = 30
TOP_REGIONS_PER_COUNTRY = 5
TOP_CITIES_PER_REGION = 5

# Set to None to use only top-N screening.
MIN_COUNTRY_SCORE = None
MIN_REGION_SCORE = None
MIN_CITY_SCORE = None

# Minimum share of criterion weight represented by available data.
# Rows below this coverage are not advanced to the next tier.
MIN_DATA_COVERAGE = 0.60

print("Screening settings loaded.")

Screening settings loaded.


In [8]:
# CELL 7 — HELPER FUNCTIONS: CRITERIA, DATA LOADING, NORMALIZATION, ENTROPY WEIGHTS

TIER_FOLDERS = {
    "tier1_country": TIER1_DIR,
    "tier2_region": TIER2_DIR,
    "tier3_city": TIER3_DIR
}

def flatten_criteria(tier_name):
    "Return {criterion_key: metadata} for one tier."
    return {
        key: details
        for category in CRITERIA[tier_name].values()
        for key, details in category.items()
    }

def load_tier_indicators(tier_name, registry):
    "Merge the available per-criterion CSVs onto a geographic registry."
    data = registry.copy()
    data["geo_id"] = data["geo_id"].astype(str)
    metadata = flatten_criteria(tier_name)
    folder = TIER_FOLDERS[tier_name]
    loaded = []

    for criterion_key in metadata:
        path = folder / f"{criterion_key}.csv"
        if not path.exists():
            continue

        indicator = pd.read_csv(path)
        required = {"geo_id", "value"}
        missing_columns = required - set(indicator.columns)
        if missing_columns:
            raise ValueError(f"{path.name} is missing columns: {missing_columns}")

        indicator["geo_id"] = indicator["geo_id"].astype(str)
        indicator[criterion_key] = pd.to_numeric(indicator["value"], errors="coerce")
        indicator = indicator[["geo_id", criterion_key]].drop_duplicates("geo_id")
        # Ignore untouched blank templates; only populated files become active criteria.
        if not indicator[criterion_key].notna().any():
            continue
        data = data.merge(indicator, on="geo_id", how="left")
        loaded.append(criterion_key)

    if not loaded:
        raise ValueError(
            f"No populated criterion CSVs found for {tier_name}. "
            f"Fill at least one template in {folder} and rerun."
        )

    return data, loaded

def normalize_series(series, direction):
    "Min-Max normalize one criterion; leave missing values missing."
    x = pd.to_numeric(series, errors="coerce")
    minimum, maximum = x.min(), x.max()

    if pd.isna(minimum) or pd.isna(maximum):
        return pd.Series(np.nan, index=x.index)

    if np.isclose(maximum, minimum):
        # A constant criterion does not distinguish places.
        return pd.Series(0.5, index=x.index).where(x.notna())

    normalized = (x - minimum) / (maximum - minimum)
    if direction == "cost":
        normalized = 1 - normalized
    return normalized.clip(0, 1)

def calculate_entropy_weights(normalized_df):
    "Entropy weights; missing normalized values are mean-imputed for weight estimation only."
    X = normalized_df.copy().astype(float)

    # Remove criteria with no observations or no variation.
    usable = []
    for col in X.columns:
        observed = X[col].dropna()
        if len(observed) >= 2 and observed.nunique() > 1:
            usable.append(col)

    if not usable:
        if len(X.columns) == 0:
            return pd.Series(dtype=float)
        return pd.Series(1 / len(X.columns), index=X.columns)

    X = X[usable].copy()
    X = X.apply(lambda col: col.fillna(col.mean()), axis=0)
    X = X.clip(lower=0)

    # Shift tiny values to avoid zero column totals.
    P = X + 1e-12
    P = P.div(P.sum(axis=0), axis=1)

    n = len(P)
    if n <= 1:
        return pd.Series(1 / len(usable), index=usable)

    k = 1 / np.log(n)
    entropy = -k * (P * np.log(P)).sum(axis=0)
    divergence = (1 - entropy).clip(lower=0)

    if divergence.sum() <= 1e-12:
        weights = pd.Series(1 / len(usable), index=usable)
    else:
        weights = divergence / divergence.sum()

    # Criteria that had no variation receive zero weight.
    final = pd.Series(0.0, index=normalized_df.columns)
    final.loc[weights.index] = weights
    return final

In [9]:
# CELL 8 — SCORE A TIER USING ENTROPY WEIGHTS

def score_tier(data, tier_name, loaded_criteria, fixed_weights=None, override=None):
    """
    Score all entities in a tier.
    override: optional tuple (criterion_key, geo_id, replacement_raw_value).
    fixed_weights: optional Series to hold weights constant during sensitivity tests.
    """
    metadata = flatten_criteria(tier_name)
    work = data.copy()

    if override is not None:
        criterion_key, target_geo_id, replacement_value = override
        if criterion_key not in loaded_criteria:
            raise ValueError(f"{criterion_key} is not loaded in {tier_name}.")
        mask = work["geo_id"].astype(str) == str(target_geo_id)
        if not mask.any():
            raise ValueError(f"geo_id {target_geo_id} was not found in this tier.")
        work.loc[mask, criterion_key] = float(replacement_value)

    normalized = pd.DataFrame(index=work.index)
    for key in loaded_criteria:
        normalized[key] = normalize_series(
            work[key], metadata[key]["direction"]
        )

    if fixed_weights is None:
        weights = calculate_entropy_weights(normalized)
    else:
        weights = fixed_weights.reindex(loaded_criteria).fillna(0.0)
        if weights.sum() <= 0:
            weights = pd.Series(1 / len(loaded_criteria), index=loaded_criteria)
        else:
            weights = weights / weights.sum()

    # A missing indicator is omitted for that entity; available weights are re-scaled.
    weighted_values = normalized.mul(weights, axis=1)
    available_weight = normalized.notna().mul(weights, axis=1).sum(axis=1)
    weighted_sum = weighted_values.sum(axis=1, min_count=1)

    work["mcda_score"] = weighted_sum / available_weight.replace(0, np.nan)
    work["data_coverage"] = available_weight / weights.sum()
    work["rank"] = work["mcda_score"].rank(method="min", ascending=False)

    return work.sort_values("mcda_score", ascending=False), weights, normalized

def keep_top_n(data, n, minimum_score=None, group_column=None):
    "Keep top N overall, or top N within each parent group."
    result = data.dropna(subset=["mcda_score"]).copy()
    if minimum_score is not None:
        result = result[result["mcda_score"] >= minimum_score]
    if group_column is None:
        return result.nlargest(n, "mcda_score")
    return (
        result.sort_values("mcda_score", ascending=False)
        .groupby(group_column, group_keys=False)
        .head(n)
    )

In [10]:
# CELL 9 — RUN TIER 1: COUNTRY SCREENING

country_data, tier1_loaded = load_tier_indicators(
    "tier1_country", country_registry
)
countries_scored, tier1_weights, tier1_normalized = score_tier(
    country_data, "tier1_country", tier1_loaded
)

countries_scored = countries_scored[
    countries_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_countries = keep_top_n(
    countries_scored,
    TOP_COUNTRIES,
    MIN_COUNTRY_SCORE
)

print("Tier 1 criteria loaded:", tier1_loaded)
print("Countries scored:", len(countries_scored))
print("Countries advanced:", len(selected_countries))
display(selected_countries[[
    "geo_id", "place_name", "mcda_score", "data_coverage", "rank"
]].head(20))
display(tier1_weights.rename("entropy_weight").to_frame())

ValueError: No populated criterion CSVs found for tier1_country. Fill at least one template in /content/data/input/tier1_country and rerun.

In [ ]:
# CELL 10 — RUN TIER 2: REGION SCREENING WITHIN SELECTED COUNTRIES

selected_country_ids = selected_countries["geo_id"].astype(str).tolist()

candidate_regions = region_registry[
    region_registry["country_id"].astype(str).isin(selected_country_ids)
].copy()

region_data, tier2_loaded = load_tier_indicators(
    "tier2_region", candidate_regions
)
regions_scored, tier2_weights, tier2_normalized = score_tier(
    region_data, "tier2_region", tier2_loaded
)

regions_scored = regions_scored[
    regions_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_regions = keep_top_n(
    regions_scored,
    TOP_REGIONS_PER_COUNTRY,
    MIN_REGION_SCORE,
    group_column="country_id"
)

print("Tier 2 criteria loaded:", tier2_loaded)
print("Candidate regions:", len(candidate_regions))
print("Regions advanced:", len(selected_regions))
display(selected_regions[[
    "geo_id", "place_name", "country_id",
    "mcda_score", "data_coverage", "rank"
]].head(30))
display(tier2_weights.rename("entropy_weight").to_frame())

In [ ]:
# CELL 11 — RUN TIER 3: CITY SCREENING WITHIN SELECTED REGIONS

selected_region_ids = selected_regions["geo_id"].astype(str).tolist()

candidate_cities = city_registry[
    city_registry["region_id"].astype(str).isin(selected_region_ids)
].copy()

city_data, tier3_loaded = load_tier_indicators(
    "tier3_city", candidate_cities
)
cities_scored, tier3_weights, tier3_normalized = score_tier(
    city_data, "tier3_city", tier3_loaded
)

cities_scored = cities_scored[
    cities_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_cities = keep_top_n(
    cities_scored,
    TOP_CITIES_PER_REGION,
    MIN_CITY_SCORE,
    group_column="region_id"
)

print("Tier 3 criteria loaded:", tier3_loaded)
print("Candidate cities:", len(candidate_cities))
print("Cities advanced:", len(selected_cities))
display(selected_cities[[
    "geo_id", "place_name", "country_id", "region_id",
    "population", "mcda_score", "data_coverage", "rank"
]].head(50))
display(tier3_weights.rename("entropy_weight").to_frame())

In [ ]:
# CELL 12 — EXPORT SCORES, SHORTLISTS, AND WEIGHTS

countries_scored.to_csv(OUTPUT_DIR / "tier1_all_country_scores.csv", index=False)
selected_countries.to_csv(OUTPUT_DIR / "tier1_selected_countries.csv", index=False)

regions_scored.to_csv(OUTPUT_DIR / "tier2_all_region_scores.csv", index=False)
selected_regions.to_csv(OUTPUT_DIR / "tier2_selected_regions.csv", index=False)

cities_scored.to_csv(OUTPUT_DIR / "tier3_all_city_scores.csv", index=False)
selected_cities.to_csv(OUTPUT_DIR / "tier3_selected_cities.csv", index=False)

pd.DataFrame({
    "tier1_country": tier1_weights,
    "tier2_region": tier2_weights,
    "tier3_city": tier3_weights
}).to_csv(OUTPUT_DIR / "entropy_weights_by_tier.csv")

print("All results exported to:", OUTPUT_DIR)

In [ ]:
# CELL 13 — USER-CONTROLLED SENSITIVITY SETTINGS

SENSITIVITY_TIER = "tier3_city"             # tier1_country / tier2_region / tier3_city
SENSITIVITY_CRITERION = "restaurants_per_10000"
SENSITIVITY_GEO_ID = "YOUR_CITY_GEONAME_ID" # replace with an ID in the scored tier

# Replace these example values with plausible values in the criterion's real unit.
TEST_VALUES = [20, 30, 40, 50, 60]

print("Sensitivity settings ready. Update the three settings and run the next cell.")

In [ ]:
# CELL 14 — RUN THE SENSITIVITY TEST

TIER_RESULTS = {
    "tier1_country": {
        "data": country_data, "loaded": tier1_loaded, "weights": tier1_weights
    },
    "tier2_region": {
        "data": region_data, "loaded": tier2_loaded, "weights": tier2_weights
    },
    "tier3_city": {
        "data": city_data, "loaded": tier3_loaded, "weights": tier3_weights
    }
}

if SENSITIVITY_TIER not in TIER_RESULTS:
    raise ValueError("SENSITIVITY_TIER must be tier1_country, tier2_region, or tier3_city.")

sens = TIER_RESULTS[SENSITIVITY_TIER]
if SENSITIVITY_CRITERION not in sens["loaded"]:
    raise ValueError(
        f"{SENSITIVITY_CRITERION} is not loaded. "
        f"Available: {sens['loaded']}"
    )

base_scored, base_weights, _ = score_tier(
    sens["data"], SENSITIVITY_TIER, sens["loaded"]
)
target_base = base_scored[
    base_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
]
if target_base.empty:
    raise ValueError("Target ID not found in this tier's scored dataset.")

baseline_value = float(
    sens["data"].loc[
        sens["data"]["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID),
        SENSITIVITY_CRITERION
    ].iloc[0]
)

sensitivity_rows = []
for test_value in TEST_VALUES:
    scenario_scored, _, _ = score_tier(
        sens["data"],
        SENSITIVITY_TIER,
        sens["loaded"],
        fixed_weights=base_weights,
        override=(SENSITIVITY_CRITERION, SENSITIVITY_GEO_ID, test_value)
    )
    target_row = scenario_scored[
        scenario_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
    ].iloc[0]

    sensitivity_rows.append({
        "tier": SENSITIVITY_TIER,
        "geo_id": SENSITIVITY_GEO_ID,
        "criterion": SENSITIVITY_CRITERION,
        "tested_value": test_value,
        "mcda_score": target_row["mcda_score"],
        "rank": target_row["rank"],
        "data_coverage": target_row["data_coverage"]
    })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

sensitivity_results.to_csv(
    OUTPUT_DIR / "sensitivity_analysis.csv", index=False
)

plt.figure(figsize=(8, 5))
plt.plot(
    sensitivity_results["tested_value"],
    sensitivity_results["mcda_score"],
    marker="o"
)
plt.xlabel(f"{SENSITIVITY_CRITERION} (raw value)")
plt.ylabel("MCDA score")
plt.title(f"Sensitivity of {SENSITIVITY_GEO_ID} to {SENSITIVITY_CRITERION}")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    sensitivity_results["tested_value"],
    sensitivity_results["rank"],
    marker="o"
)
plt.xlabel(f"{SENSITIVITY_CRITERION} (raw value)")
plt.ylabel("Rank (1 = best)")
plt.title(f"Rank sensitivity of {SENSITIVITY_GEO_ID}")
plt.gca().invert_yaxis()
plt.grid(True)
plt.show()

In [ ]:
# CELL 14 — RUN THE SENSITIVITY TEST

TIER_RESULTS = {
    "tier1_country": {
        "data": country_data, "loaded": tier1_loaded, "weights": tier1_weights
    },
    "tier2_region": {
        "data": region_data, "loaded": tier2_loaded, "weights": tier2_weights
    },
    "tier3_city": {
        "data": city_data, "loaded": tier3_loaded, "weights": tier3_weights
    }
}

if SENSITIVITY_TIER not in TIER_RESULTS:
    raise ValueError("SENSITIVITY_TIER must be tier1_country, tier2_region, or tier3_city.")

sens = TIER_RESULTS[SENSITIVITY_TIER]
if SENSITIVITY_CRITERION not in sens["loaded"]:
    raise ValueError(
        f"{SENSITIVITY_CRITERION} is not loaded. "
        f"Available: {sens['loaded']}"
    )

base_scored, base_weights, _ = score_tier(
    sens["data"], SENSITIVITY_TIER, sens["loaded"]
)
target_base = base_scored[
    base_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
]
if target_base.empty:
    raise ValueError("Target ID not found in this tier's scored dataset.")

baseline_value = float(
    sens["data"].loc[
        sens["data"]["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID),
        SENSITIVITY_CRITERION
    ].iloc[0]
)

sensitivity_rows = []
for test_value in TEST_VALUES:
    scenario_scored, _, _ = score_tier(
        sens["data"],
        SENSITIVITY_TIER,
        sens["loaded"],
        fixed_weights=base_weights,
        override=(SENSITIVITY_CRITERION, SENSITIVITY_GEO_ID, test_value)
    )
    target_row = scenario_scored[
        scenario_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
    ].iloc[0]

    sensitivity_rows.append({
        "tier": SENSITIVITY_TIER,
        "geo_id": SENSITIVITY_GEO_ID,
        "criterion": SENSITIVITY_CRITERION,
        "tested_value": test_value,
        "mcda_score": target_row["mcda_score"],
        "rank": target_row["rank"],
        "data_coverage": target_row["data_coverage"]
    })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

sensitivity_results.to_csv(
    OUTPUT_DIR / "sensitivity_analysis.csv", index=False
)

plt.figure(figsize=(8, 5))
plt.plot(
    sensitivity_results["tested_value"],
    sensitivity_results["mcda_score"],
    marker="o"
)
plt.xlabel(f"{SENSITIVITY_CRITERION} (raw value)")
plt.ylabel("MCDA score")
plt.title(f"Sensitivity of {SENSITIVITY_GEO_ID} to {SENSITIVITY_CRITERION}")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    sensitivity_results["tested_value"],
    sensitivity_results["rank"],
    marker="o"
)
plt.xlabel(f"{SENSITIVITY_CRITERION} (raw value)")
plt.ylabel("Rank (1 = best)")
plt.title(f"Rank sensitivity of {SENSITIVITY_GEO_ID}")
plt.gca().invert_yaxis()
plt.grid(True)
plt.show()